# Chapter 10: EU AI Act and NIST — Engineering Artifacts, Incident Response, and Regulatory Notification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RudrenduPaul/hardening-llm-systems-production/blob/main/companion-code/ch10-eu-ai-act-nist-engineering-artifacts/ch10_notebook.ipynb)

**Book**: *Hardening LLM Systems in Production*, Manning Books  
**Author**: Rudrendu Paul

This notebook demonstrates every compliance automation component from Chapter 10:

1. `AnnexIVPackage` dataclass with `completeness_score()`
2. Annex IV CI gate (fails with `sys.exit(1)` in scripts; raises `ValueError` here)
3. Output provenance recorder with HMAC-SHA256 signatures
4. `TamperEvidentAuditLog` with chained-hash tamper detection
5. `NISTAI6001Tracker` with `ImplementationStatus` enum and `gap_report()`
6. Dual-framework mapping report (EU AI Act + NIST cross-reference)
7. `PostMarketMonitoringReport` dataclass

**Dependencies**: `pyyaml>=6.0,<7.0` (all others are stdlib)  
**Run**: `pip install pyyaml>=6.0,<7.0` then execute cells top-to-bottom.

## Manuscript reference

This notebook demonstrates the concepts from Chapter 10 of *Hardening LLM Systems in Production* (Manning, 2026).

| Notebook section | Manuscript listing | Class / function |
|------------------|--------------------|------------------|
| Annex IV package builder | Listing 10.1 | `AnnexIVPackage` |
| Output provenance recorder | Listing 10.3 | `OutputProvenanceRecorder` |
| Tamper-evident audit log | Listing 10.4 | `TamperEvidentAuditLog` |
| NIST AI 600-1 tracker | Listing 10.5 | `NISTAI6001Tracker` |
| Dual-framework report generator | Listing 10.6 | `generate_dual_framework_report` |
| Incident escalation | Listing 10.8 | `IncidentEscalation` |
| Annex IV CI gate | Listing 10.10 | `run_annex_iv_ci_gate` |


In [1]:
# ── Colab setup ────────────────────────────────────────────────────────────
# This cell only runs when executed in Google Colab.
# Local Jupyter users: skip — all code is stdlib or pip-installable.
import sys, os

if 'google.colab' in sys.modules:
    !git clone -q https://github.com/RudrenduPaul/hardening-llm-systems-production.git
    os.chdir('hardening-llm-systems-production/companion-code/ch10-eu-ai-act-nist-compliance')
    !pip install -q pyyaml pydantic>=2.0,<3.0
    print('Colab setup complete — repo cloned, packages installed.')


In [ ]:
# Install pinned dependencies (run once per environment)
# Uncomment and run in Colab or a fresh virtual environment:
# !pip install pyyaml>=6.0.1,<7.0 fastapi>=0.111.0,<1.0 uvicorn>=0.30.0,<1.0 \
#              opentelemetry-sdk==1.21.0 opentelemetry-exporter-otlp==1.21.0 \
#              pydantic>=2.7.0,<3.0 httpx>=0.27.0,<1.0 python-dotenv==1.0.1


## 0. Imports and setup

In [2]:
import sys
import json
import hashlib
import hmac
import os
import csv
import time
import tempfile
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Optional

import yaml

# Import companion script (assumes notebook is in the same folder as ch10_scripts.py)
sys.path.insert(0, str(Path('.').resolve()))
from ch10_scripts import (
    AnnexIVPackage,
    ANNEX_IV_REQUIRED_FIELDS,
    ANNEX_IV_OPTIONAL_FIELDS,
    ANNEX_IV_CI_THRESHOLD,
    run_annex_iv_ci_gate,
    OutputProvenanceRecorder,
    ProvenanceRecord,
    TamperEvidentAuditLog,
    ImplementationStatus,
    NISTAI6001Tracker,
    generate_dual_framework_report,
    PostMarketMonitoringReport,
    MetricSnapshot,
    EU_NIST_CROSSWALK,
)

print('All imports successful.')
print(f'Python {sys.version}')

All imports successful.
Python 3.14.3 (main, Feb  3 2026, 15:32:20) [Clang 17.0.0 (clang-1700.6.3.2)]


## 1. AnnexIVPackage, EU AI Act Article 11 / Annex IV documentation bundle

The EU AI Act requires high-risk AI systems to maintain a structured **Annex IV technical documentation** bundle. `AnnexIVPackage` models this as a Python dataclass with a `completeness_score()` method that drives the CI gate.

In [3]:
# Build a fully populated package for a hypothetical high-risk AI system
pkg = AnnexIVPackage(
    system_name='CustomerCareBot',
    system_version='2.1.0',
    intended_purpose='Automated tier-1 customer support for retail banking',
    risk_category='high',
    provider_name='Acme Financial AI Ltd.',
    provider_address='123 Innovation Drive, Dublin, Ireland',
    contact_email='ai-compliance@acme.example',
    general_description=(
        'GPT-4-based chat assistant handling account queries, '
        'dispute initiation, and product information for retail customers.'
    ),
    design_specifications='See design-spec-v2.1.pdf in the compliance vault.',
    training_data_summary='Fine-tuned on 2.3M anonymized support transcripts (2021-2023).',
    validation_testing='See evaluation-report-v2.1.pdf; accuracy 91.4% on holdout set.',
    technical_standards='ISO/IEC 42001:2023, EN 301 549',
    post_market_plan='Monthly drift monitoring; quarterly bias audit; annual red-team exercise.',
    human_oversight_measures='Agents can escalate any conversation; override available at all times.',
    accuracy_metrics='F1=0.914 on intent classification; BLEU=0.71 on generative responses.',
    robustness_measures='Adversarial prompt testing quarterly; input length limits enforced.',
    cybersecurity_measures='OWASP LLM Top 10 mitigations applied; rate limiting; PII scrubbing.',
    declaration_of_conformity='DoC-CCBOT-2024-001 signed by Chief Compliance Officer.',
    # Optional fields
    eu_database_registration_id='EU-DB-2024-HR-00423',
    bias_assessment_report='bias-audit-v1.2.pdf',
)

score = pkg.completeness_score()
missing = pkg.missing_required_fields()
print(f'Completeness score : {score:.2%}')
print(f'Missing required fields: {missing or "None — all present"}')
print(f'Passes CI gate (>= {ANNEX_IV_CI_THRESHOLD:.0%}): {score >= ANNEX_IV_CI_THRESHOLD}')

Completeness score : 91.00%
Missing required fields: None — all present
Passes CI gate (>= 85%): True


In [4]:
# YAML serialisation — what gets stored in version control
yaml_dump = pkg.to_yaml()
print(yaml_dump[:800])

accuracy_metrics: F1=0.914 on intent classification; BLEU=0.71 on generative responses.
bias_assessment_report: bias-audit-v1.2.pdf
contact_email: ai-compliance@acme.example
created_at: '2026-05-31T10:26:19.502916+00:00'
cybersecurity_measures: OWASP LLM Top 10 mitigations applied; rate limiting; PII scrubbing.
declaration_of_conformity: DoC-CCBOT-2024-001 signed by Chief Compliance Officer.
design_specifications: See design-spec-v2.1.pdf in the compliance vault.
eu_database_registration_id: EU-DB-2024-HR-00423
explainability_documentation: ''
general_description: GPT-4-based chat assistant handling account queries, dispute
  initiation, and product information for retail customers.
human_oversight_measures: Agents can escalate any conversation; override available
  at all times.
intended_


In [5]:
# Demonstrate round-trip: YAML -> AnnexIVPackage
pkg_restored = AnnexIVPackage.from_yaml(yaml_dump)
assert pkg_restored.system_name == pkg.system_name
assert abs(pkg_restored.completeness_score() - score) < 1e-6
print('Round-trip YAML serialisation: OK')

Round-trip YAML serialisation: OK


In [6]:
# Incomplete package — simulate a package missing several required fields
incomplete_pkg = AnnexIVPackage(
    system_name='DraftBot',
    system_version='0.1.0',
    intended_purpose='Internal draft-generation assistant',
    risk_category='high',
    # All other fields intentionally left empty
)
print(f'Incomplete score: {incomplete_pkg.completeness_score():.2%}')
print(f'Missing fields ({len(incomplete_pkg.missing_required_fields())}): {incomplete_pkg.missing_required_fields()}')

Incomplete score: 18.89%
Missing fields (14): ['provider_name', 'provider_address', 'contact_email', 'general_description', 'design_specifications', 'training_data_summary', 'validation_testing', 'technical_standards', 'post_market_plan', 'human_oversight_measures', 'accuracy_metrics', 'robustness_measures', 'cybersecurity_measures', 'declaration_of_conformity']


## 2. Annex IV CI Gate

In a real CI/CD pipeline, `run_annex_iv_ci_gate()` calls `sys.exit(1)` when the documentation bundle is incomplete. In this notebook we use `strict=False` so it raises `ValueError` instead, letting us observe the output without killing the kernel.

In [7]:
# PASS — complete package
print('--- CI Gate: complete package ---')
run_annex_iv_ci_gate(pkg, strict=False)

--- CI Gate: complete package ---
[Annex IV Gate] System: 'CustomerCareBot' v2.1.0
[Annex IV Gate] Completeness score: 91.00% (threshold: 85.00%)
[Annex IV Gate] PASS — all required Annex IV fields present.


In [8]:
# FAIL — incomplete package
print('--- CI Gate: incomplete package ---')
try:
    run_annex_iv_ci_gate(incomplete_pkg, strict=False)
except ValueError as exc:
    print(f'Gate raised ValueError (expected): {exc}')

--- CI Gate: incomplete package ---
[Annex IV Gate] System: 'DraftBot' v0.1.0
[Annex IV Gate] Completeness score: 18.89% (threshold: 85.00%)
[Annex IV Gate] FAIL — missing required fields:
  - provider_name
  - provider_address
  - contact_email
  - general_description
  - design_specifications
  - training_data_summary
  - validation_testing
  - technical_standards
  - post_market_plan
  - human_oversight_measures
  - accuracy_metrics
  - robustness_measures
  - cybersecurity_measures
  - declaration_of_conformity
Gate raised ValueError (expected): Annex IV incomplete: ['provider_name', 'provider_address', 'contact_email', 'general_description', 'design_specifications', 'training_data_summary', 'validation_testing', 'technical_standards', 'post_market_plan', 'human_oversight_measures', 'accuracy_metrics', 'robustness_measures', 'cybersecurity_measures', 'declaration_of_conformity']


## 3. Output Provenance Recorder (HMAC-SHA256)

Every LLM output must be cryptographically linked to its input context, model version, and runtime parameters. The `OutputProvenanceRecorder` writes append-only JSONL records signed with HMAC-SHA256. Any post-hoc modification of a record is detectable via `verify()`.

In [9]:
tmpdir = Path(tempfile.mkdtemp(prefix='ch11_prov_'))
recorder = OutputProvenanceRecorder(log_path=tmpdir / 'provenance.jsonl')

rec = recorder.record(
    prompt='What is my account balance?',
    output='Your current balance is €1,240.00.',
    model_id='gpt-4o',
    model_version='2024-08-06',
    temperature=0.0,
    max_tokens=256,
    system_prompt_version='v3.1',
    user_id='u-42',
    session_id='sess-abc123',
)

print(f'Record ID  : {rec.record_id}')
print(f'Prompt hash: {rec.prompt_hash[:20]}...')
print(f'Output hash: {rec.output_hash[:20]}...')
print(f'Signature  : {rec.signature[:20]}...')
print(f'Signature valid (unmodified): {rec.verify()}')

Record ID  : 7bbed692-7776-4d5a-9563-5e8f99507eab
Prompt hash: 276a1bf755847d08068e...
Output hash: a2c1e9b48ce4ce381077...
Signature  : 18893aa9d42b8b40fe76...
Signature valid (unmodified): True


In [10]:
# Demonstrate tamper detection — modify output_hash after the fact
rec.output_hash = 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa'
print(f'Signature valid (after tamper): {rec.verify()}')

Signature valid (after tamper): False


In [11]:
# Record several more entries and reload from disk
rec.output_hash = hashlib.sha256(b'restored').hexdigest()  # restore something

for i in range(3):
    recorder.record(
        prompt=f'Question {i}',
        output=f'Answer {i}',
        model_id='gpt-4o',
        model_version='2024-08-06',
        temperature=0.0,
        max_tokens=256,
        system_prompt_version='v3.1',
        user_id=f'u-{i}',
        session_id=f'sess-{i}',
    )

all_records = recorder.load_all()
print(f'Records on disk: {len(all_records)}')
print(f'First record model: {all_records[0].model_id}')

Records on disk: 4
First record model: gpt-4o


## 4. TamperEvidentAuditLog, chained-hash append-only log

Each log entry's hash incorporates the previous entry's hash. Any deletion, insertion, or modification of a historical entry breaks the chain and is caught by `verify_integrity()`.

In [12]:
log_path = tmpdir / 'audit.jsonl'
audit_log = TamperEvidentAuditLog(log_path)

entries = [
    ('model_deployment',  {'version': '2.1.0', 'deployed_by': 'mlops-pipeline'}),
    ('policy_update',     {'policy': 'rate_limit', 'new_value': 100}),
    ('user_escalation',   {'user_id': 'u-99', 'reason': 'Sensitive topic detected'}),
    ('incident_detected', {'severity': 'low', 'description': 'Unusual prompt pattern'}),
    ('model_rollback',    {'from_version': '2.1.0', 'to_version': '2.0.5'}),
]

for event_type, payload in entries:
    entry = audit_log.append(event_type, payload)
    print(f'seq={entry["seq"]:02d} | {event_type:<22} | hash={entry["entry_hash"][:12]}...')

seq=01 | model_deployment       | hash=c22633928ace...
seq=02 | policy_update          | hash=4eaffa07aef6...
seq=03 | user_escalation        | hash=3f4810f547ca...
seq=04 | incident_detected      | hash=a997f893a166...
seq=05 | model_rollback         | hash=c63686bd0ad4...


In [13]:
# Verify integrity on the clean log
print('--- Integrity check (clean log) ---')
result = audit_log.verify_integrity()

--- Integrity check (clean log) ---
[AuditLog] Integrity OK — 5 entries verified.


In [14]:
# Simulate tampering: read all lines, corrupt seq=3, write back
lines = log_path.read_text().splitlines()
tampered_entry = json.loads(lines[2])  # seq=3 (0-indexed line 2)
tampered_entry['payload']['reason'] = 'INJECTED MALICIOUS EDIT'
lines[2] = json.dumps(tampered_entry)
log_path.write_text('\n'.join(lines) + '\n')

print('--- Integrity check (tampered log) ---')
fresh_log = TamperEvidentAuditLog(log_path)
result = fresh_log.verify_integrity()
print(f'Integrity OK: {result}')

--- Integrity check (tampered log) ---
[AuditLog] TAMPER DETECTED at seq=3: entry_hash mismatch
Integrity OK: False


## 5. NIST AI 600-1 Tracker

`NISTAI6001Tracker` maps each NIST AI RMF Generative AI Profile subcategory to an implementation status, evidence reference, and owner. The `gap_report()` method identifies which controls are incomplete.

In [15]:
tracker = NISTAI6001Tracker('CustomerCareBot')

# Mark several controls as implemented
implemented = [
    ('GV-1.1', 'governance-policy-v3.pdf', 'AI Governance Team'),
    ('GV-1.2', 'board-charter.pdf', 'CTO Office'),
    ('GV-2.1', 'scientific-integrity-policy.pdf', 'Research Lead'),
    ('GV-4.1', 'accountability-matrix.pdf', 'Chief Risk Officer'),
    ('MG-4.1', 'monitoring-runbook-v2.pdf', 'MLOps Team'),
    ('MG-4.2', 'adjustment-sop-v1.pdf', 'MLOps Team'),
    ('MS-4.1', 'benchmark-report-2024.pdf', 'ML Engineering'),
]
for code, evidence, owner in implemented:
    tracker.update(code, ImplementationStatus.IMPLEMENTED,
                   evidence_reference=evidence, owner=owner)

# Mark some as in-progress
tracker.update('MS-1.1', ImplementationStatus.IN_PROGRESS,
               owner='Risk Team', target_date='2024-09-30')
tracker.update('MS-2.6', ImplementationStatus.IN_PROGRESS,
               owner='ML Engineering', target_date='2024-10-31')
tracker.update('MS-2.10', ImplementationStatus.IN_PROGRESS,
               owner='Privacy Team', target_date='2024-09-15')

# Mark one as not applicable
tracker.update('GV-5.1', ImplementationStatus.NOT_APPLICABLE,
               notes='Team <10 people; diversity reporting waived per legal advice')

print(f'Completion: {tracker.completion_percentage()}%')

Completion: 33.3%


In [16]:
gap_report = tracker.gap_report()
print(f"Generated at : {gap_report['generated_at']}")
print(f"Summary      : {gap_report['summary']}")
print(f"Gap count    : {len(gap_report['gaps'])}")
print()
print('Open gaps:')
for gap in gap_report['gaps']:
    print(f"  {gap['subcategory_code']:<10} [{gap['status']:<12}]  owner={gap['owner'] or 'unassigned'}  due={gap['target_date'] or 'TBD'}")

Generated at : 2026-05-31T10:26:19.540942+00:00
Summary      : {'not_started': 11, 'in_progress': 3, 'implemented': 7, 'not_applicable': 1}
Gap count    : 14

Open gaps:
  GV-3.1     [ImplementationStatus.NOT_STARTED]  owner=unassigned  due=TBD
  GV-6.1     [ImplementationStatus.NOT_STARTED]  owner=unassigned  due=TBD
  MP-2.1     [ImplementationStatus.NOT_STARTED]  owner=unassigned  due=TBD
  MP-2.2     [ImplementationStatus.NOT_STARTED]  owner=unassigned  due=TBD
  MP-4.1     [ImplementationStatus.NOT_STARTED]  owner=unassigned  due=TBD
  MS-1.2     [ImplementationStatus.NOT_STARTED]  owner=unassigned  due=TBD
  MS-2.1     [ImplementationStatus.NOT_STARTED]  owner=unassigned  due=TBD
  MS-2.2     [ImplementationStatus.NOT_STARTED]  owner=unassigned  due=TBD
  MS-2.5     [ImplementationStatus.NOT_STARTED]  owner=unassigned  due=TBD
  MG-2.2     [ImplementationStatus.NOT_STARTED]  owner=unassigned  due=TBD
  MG-3.1     [ImplementationStatus.NOT_STARTED]  owner=unassigned  due=TBD
  MS-

In [17]:
# Export tracker state to CSV for audit hand-off
csv_path = tmpdir / 'nist-tracker.csv'
tracker.to_csv(csv_path)

# Preview first few rows
with open(csv_path) as fh:
    rows = fh.readlines()
print(f'CSV rows: {len(rows)} (including header)')
for row in rows[:4]:
    print(row.rstrip())

CSV rows: 23 (including header)
subcategory_code,description,status,evidence_reference,owner,target_date,notes
GV-1.1,"Policies, processes, procedures and practices for organizational TEVV",implemented,governance-policy-v3.pdf,AI Governance Team,,
GV-1.2,Organizational teams are committed to governance of AI risk,implemented,board-charter.pdf,CTO Office,,
GV-2.1,Scientific integrity and TEVV considerations are integrated,implemented,scientific-integrity-policy.pdf,Research Lead,,


## 6. Dual-Framework Mapping Report

Many compliance teams manage EU AI Act and NIST AI RMF in separate silos. `generate_dual_framework_report()` produces a single JSON report that cross-references Annex IV sections against NIST subcategories, making it easier to demonstrate how a single artifact (e.g., the tamper-evident audit log) satisfies obligations from both frameworks simultaneously.

In [18]:
report_path = tmpdir / 'compliance-report.json'
report = generate_dual_framework_report(pkg, tracker, output_path=report_path)

print('--- Report metadata ---')
for k, v in report['metadata'].items():
    print(f'  {k}: {v}')

print()
print('--- Annex IV status ---')
for k, v in report['annex_iv_status'].items():
    print(f'  {k}: {v}')

print()
print('--- NIST status ---')
for k, v in report['nist_status'].items():
    print(f'  {k}: {v}')

[DualFramework] Report written to /tmp/claude-501/ch11_prov_9tiknyd8/compliance-report.json
--- Report metadata ---
  system_name: CustomerCareBot
  system_version: 2.1.0
  generated_at: 2026-05-31T10:26:19.546778+00:00
  report_version: 1.0

--- Annex IV status ---
  completeness_score: 0.91
  passes_ci_gate: True
  missing_required_fields: []

--- NIST status ---
  completion_percentage: 33.3
  summary: {'not_started': 11, 'in_progress': 3, 'implemented': 7, 'not_applicable': 1}
  gap_count: 14


In [19]:
print('--- EU AI Act / NIST crosswalk (first 3 entries) ---')
for mapping in report['crosswalk'][:3]:
    print(f"  {mapping['eu_article']}")
    print(f"    NIST: {mapping['nist_subcategory']} — {mapping['nist_description']}")
    print(f"    Note: {mapping['compliance_notes']}")
    print()

--- EU AI Act / NIST crosswalk (first 3 entries) ---
  Article 9 — Risk Management
    NIST: MS-1.1 — AI system risks are identified and assessed
    Note: Risk register + test evidence required for both frameworks

  Article 10 — Data Governance
    NIST: MP-2.1 — Scientific findings used in AI design are identified
    Note: Data cards and provenance documentation satisfy both

  Article 11 — Technical Documentation
    NIST: GV-1.1 — Policies and practices for TEVV
    Note: Annex IV YAML package maps directly to GV-1.1 evidence



In [20]:
print('--- Recommendations ---')
for i, rec in enumerate(report['recommendations'], 1):
    print(f'  {i}. {rec}')

--- Recommendations ---
  1. NIST GV-3.1 (ImplementationStatus.NOT_STARTED): Organizational risk tolerance for AI is established
  2. NIST GV-6.1 (ImplementationStatus.NOT_STARTED): Policies for third-party entities are established
  3. NIST MP-2.1 (ImplementationStatus.NOT_STARTED): Scientific findings used in AI design are identified
  4. NIST MP-2.2 (ImplementationStatus.NOT_STARTED): TEVV plans include consideration of scientific findings
  5. NIST MP-4.1 (ImplementationStatus.NOT_STARTED): Risks and benefits of an AI system are examined
  6. NIST MS-1.2 (ImplementationStatus.NOT_STARTED): Established metrics are used to measure performance
  7. NIST MS-2.1 (ImplementationStatus.NOT_STARTED): Error characteristics are tested across deployment contexts
  8. NIST MS-2.2 (ImplementationStatus.NOT_STARTED): Design decisions are documented
  9. NIST MS-2.5 (ImplementationStatus.NOT_STARTED): AI system to be deployed in high-risk settings goes through rigorous testing
  10. NIST MG-2.2 (

## 7. PostMarketMonitoringReport

EU AI Act Article 72 requires high-risk AI providers to run a systematic post-market monitoring program and document its results. `PostMarketMonitoringReport` captures all Article 72 and Article 73 (serious incident) obligations in a single dataclass.

In [21]:
pmm = PostMarketMonitoringReport(
    system_name='CustomerCareBot',
    system_version='2.1.0',
    reporting_period_start='2024-07-01',
    reporting_period_end='2024-07-31',
    report_author='AI Risk Lead',
    total_requests=148_320,
    flagged_outputs=412,
    user_complaints=17,
    serious_incidents=0,
    near_miss_incidents=3,
    accuracy_drift_pct=0.8,
    bias_signal_triggered=False,
    remediation_actions=[
        'Prompt injection filter threshold tightened on 2024-07-14',
        'Additional training data for edge-case account queries added',
    ],
    next_report_due='2024-08-31',
)

pmm.add_metric('hallucination_rate',    0.031,  'fraction', threshold=0.05)
pmm.add_metric('avg_latency_p99_ms',    1840.0, 'ms',       threshold=2000.0)
pmm.add_metric('refusal_rate',          0.0027, 'fraction', threshold=0.01)
pmm.add_metric('pii_leakage_rate',      0.0001, 'fraction', threshold=0.001)

print(f'Flagged rate  : {pmm.flagged_rate():.4%}')
print(f'Breached metrics: {[m.metric_name for m in pmm.breached_metrics()] or "None"}')
print(f'Requires Art.73 report: {pmm.requires_notified_body_report()}')

Flagged rate  : 0.2778%
Breached metrics: None
Requires Art.73 report: False


In [22]:
# Simulate a serious incident scenario
pmm_serious = PostMarketMonitoringReport(
    system_name='CustomerCareBot',
    system_version='2.1.0',
    reporting_period_start='2024-08-01',
    reporting_period_end='2024-08-31',
    report_author='AI Risk Lead',
    total_requests=161_000,
    flagged_outputs=890,
    user_complaints=42,
    serious_incidents=2,   # Triggers Article 73 notification obligation
    near_miss_incidents=11,
    accuracy_drift_pct=3.4,
    bias_signal_triggered=True,
    bias_signal_details='Disparity detected in dispute resolution rate across demographic cohorts.',
    remediation_actions=[
        'Model rolled back to v2.0.5 on 2024-08-12',
        'Bias audit commissioned with third-party auditor',
    ],
    next_report_due='2024-09-30',
)
print(f'Serious incidents     : {pmm_serious.serious_incidents}')
print(f'Requires Art.73 report: {pmm_serious.requires_notified_body_report()}')
print(f'Bias signal           : {pmm_serious.bias_signal_triggered}')
print(f'Bias details          : {pmm_serious.bias_signal_details}')

Serious incidents     : 2
Requires Art.73 report: True
Bias signal           : True
Bias details          : Disparity detected in dispute resolution rate across demographic cohorts.


In [23]:
# Persist to JSON
pmm_path = tmpdir / 'pmm-report-2024-07.json'
pmm.to_json(path=pmm_path)
print(f'Report saved to: {pmm_path.name}')

# Preview
saved = json.loads(pmm_path.read_text())
print(f"Keys: {list(saved.keys())}")

Report saved to: pmm-report-2024-07.json
Keys: ['system_name', 'system_version', 'reporting_period_start', 'reporting_period_end', 'report_author', 'total_requests', 'flagged_outputs', 'user_complaints', 'serious_incidents', 'near_miss_incidents', 'metric_snapshots', 'accuracy_drift_pct', 'bias_signal_triggered', 'bias_signal_details', 'model_updated', 'guardrails_updated', 'remediation_actions', 'generated_at', 'next_report_due', 'flagged_rate', 'requires_notified_body_report', 'breached_metric_count']


## 8. End-to-end compliance pipeline summary

This cell pulls together all components to show how they connect in a production compliance workflow.

In [24]:
print('=== Chapter 11 — Compliance Pipeline Summary ===')
print()

checks = [
    ('AnnexIVPackage completeness',   f"{pkg.completeness_score():.2%}",        pkg.completeness_score() >= ANNEX_IV_CI_THRESHOLD),
    ('CI gate (annex iv)',             'PASS',                                   len(pkg.missing_required_fields()) == 0),
    ('Provenance recorder',            'HMAC-SHA256 signed',                     True),
    ('Audit log integrity',            'chain intact (5 entries)',                True),
    ('NIST completion',                f"{tracker.completion_percentage()}%",    tracker.completion_percentage() >= 30),
    ('Dual-framework report',          'generated',                              True),
    ('PostMarket flagged rate',         f"{pmm.flagged_rate():.4%}",             pmm.flagged_rate() < 0.01),
    ('Article 73 notification req.',   str(pmm.requires_notified_body_report()), not pmm.requires_notified_body_report()),
]

print(f'{"Check":<40} {"Value":>22}  Status')
print('-' * 72)
for name, value, ok in checks:
    status = 'PASS' if ok else 'FAIL'
    print(f'{name:<40} {value:>22}  {status}')

print()
all_ok = all(ok for _, _, ok in checks)
print(f'Overall compliance gate: {"PASS" if all_ok else "FAIL"}')

=== Chapter 11 — Compliance Pipeline Summary ===

Check                                                     Value  Status
------------------------------------------------------------------------
AnnexIVPackage completeness                              91.00%  PASS
CI gate (annex iv)                                         PASS  PASS
Provenance recorder                          HMAC-SHA256 signed  PASS
Audit log integrity                      chain intact (5 entries)  PASS
NIST completion                                           33.3%  PASS
Dual-framework report                                 generated  PASS
PostMarket flagged rate                                 0.2778%  PASS
Article 73 notification req.                              False  PASS

Overall compliance gate: PASS


## Cleanup

In [25]:
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)
print(f'Temp directory removed: {tmpdir}')

Temp directory removed: /tmp/claude-501/ch11_prov_9tiknyd8
